In [14]:
from langchain_core.messages import AIMessage
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_core.documents import Document


# =============1.准备文档================
with open("./resources/评估.md", encoding="utf-8") as f:
    md_content = f.read()

# =============2.切分文档================
# 结构感知切分器
spliter = MarkdownHeaderTextSplitter(
    # 切分依据，这里是按照三级标题
    headers_to_split_on = [
        ("#", "h1"),
        ("##", "h2"),
        ("###", "h3")
    ], strip_headers=True
)
# 执行切分
docs: list[Document] = spliter.split_text(md_content)

# 准备向量模型
ollama_embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",  # 性价比高的模型
    dimensions=1024
)
# 初始化向量数据库
vector_store = InMemoryVectorStore(embedding=ollama_embeddings)

In [15]:
# 保存文档到向量数据库
ids = vector_store.add_documents(docs)

print(ids)

['53ff6fe9-ac19-446d-b17b-a02f4280c928', 'ac607917-22e9-4b3b-bb95-143b42502da0', '2a926852-d0ec-4e86-a327-3f8dffee73f9', '4a7a36f5-1c4d-452e-a17d-03be632240ef', 'de59a191-7a93-43f3-8364-fe163690e9a6', 'f0b1e9e6-8e87-47b1-96ab-fd52d087c4eb', 'da4cf438-efb2-4e2b-b847-a7428b2fbe92', '0d29ac66-c7d3-42fe-8874-33316842e11d', '50595a35-32fd-4cc7-82d7-1cac6df3fbfa', '48a733b0-a5b4-4c0f-a816-e25239f156c9', '5a333d8b-f5c7-4867-a83b-e333b2f953b6', 'c4daa7e8-29e0-48b8-acd5-5e11ce7eeba4', 'db3c69e2-0392-47d8-a92e-8719994d241e', '935b1914-caf2-4246-a4bf-1b80ea13ee82', '4acfa566-d1d6-4305-9414-2c8b451724bb', 'a0e49e4a-5491-49e9-8755-eb5198c7d399', '52152856-38c9-46c0-8ce8-abe6fa93fa23', 'e17e389d-13b1-4216-92c5-fc77de0a62f4', 'af1620de-57cd-4f61-b84a-2da877d427e2', 'c95a34f5-4b62-4465-a5b3-3a56ece46e03', 'b2662308-832e-4be1-96f0-56c1be796127']


In [ ]:
# 删除旧文档
vector_store.delete(["b2662308-832e-4be1-96f0-56c1be796127"])

In [ ]:
from langchain_core.documents import Document

# 执行相似度检索 - 无得分
doc_list: list[Document] = vector_store.similarity_search(
    query="论语中教育的目的是什么?",
    k=5
)

for index, doc in enumerate(doc_list, start=1):
    print(f"===============第{index}个文档")
    print(f"元数据\n{doc.metadata}")
    print(f"文本数据{doc.page_content}")

In [ ]:
# 相似度检索 - 有得分
doc_list: list[tuple[Document, float]] = vector_store.similarity_search_with_score(
    query="论语中教育的目的是什么",
    k=5
)
for doc in doc_list:
    print(type(doc))
    document = doc[0]
    score = doc[1]
    print(f"=======================得分: {score}")
    print(f"元数据: {document.metadata}")
    print(f"文本数据: {document.page_content}")

In [19]:
from langchain.chat_models import init_chat_model
from app.core.config import settings

# 知识检索


# 准备问题
query ="论语中教育的目的是什么"

# 向量数据库检索
doc_list: list[Document] = vector_store.similarity_search(query=query, k=5)

# 增强提示词
message = f"""
"你是一个教资考试专家，你需要根据提供的上下文回答用户的问题，
不能自己编造答案。答案要简短，直接引用原文。
用户的问题： {query}
增强知识：
{[doc.page_content for doc in doc_list]}
"""
print(f"提示词\n{message}")

# 准备大模型
model = init_chat_model(
    model="deepseek-v4-flash",
    api_key=settings.llm.api_key,
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)

# 调用大模型
ai_message: AIMessage = model.invoke(message)
ai_message.pretty_print()

提示词

"你是一个教资考试专家，你需要根据提供的上下文回答用户的问题，
不能自己编造答案。答案要简短，直接引用原文。
用户的问题： 论语中教育的目的是什么
增强知识：
['**人性论**：人性本善，人先天具有仁、义、礼、智四个“善端”。  \n**教育作用**：发扬善端，培养道德完人，得天下英才而教育之。  \n**教学原则**：循序渐进，专心有恒。', '**人性论**：性恶论，化性起伪，善德是后天习得的，重视教育的作用。  \n**教学原则**：学以致用，锲而不舍。', '国内：《学记》，世界最早，成文于战国末期，作者是乐正克。  \n国外：昆体良《雄辩术原理》/《论演说家的教育》。', '**教育作用**：庶、富、教；性相近，习相远。  \n**教育对象**：“有教无类”，教育民主思想。  \n**教育目的**：以完善人格为教育的首要目的，培养士和君子。  \n**教学内容**：文、行、忠、义。  \n**教学过程**：学、思、习、行。  \n**教学原则**：因材施教、启发诱导、学思结合、谦虚笃实。  \n**教师观**：其身正，不令而行；其身不正，虽令不从。', '**教育作用**：化民成俗，其必由学；建国君民，教学为先。  \n**教学相长**：教和学两方面互相影响和促进，都得到提高。  \n**豫时孙摩**：包括预防性原则、及时性原则、循序渐进原则、集体教育原则。  \n**长善救失**：“学者有四失，教者必知之。人之学也，或失则多，或失则寡，或失则易，或失则止。此四者，心之莫同也。知其心，然后能救其失也，教也者，长善而救其失者也。”  \n**启发诱导**：道而弗牵，强而弗抑，开而弗达。']

================================== Ai Message ==================================

以完善人格为教育的首要目的，培养士和君子。
